<a href="https://colab.research.google.com/github/donna6355/study_python/blob/master/senior/csv_cluster.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import pandas as pd
df = pd.read_csv('merged_result.csv')
df

,title,contents,date,token
0,일월도서관 5060세대를 위한 인문학과 독서,프로그램명 세대를 위한 인문학과 독서 교육기간 매주 화 ...,2026. 2. 6. 15:16,"['프로그램', '세대', '위', '인문학', '독서', '교육', '기간', '..."
1,5060 핸드폰 화면 어두울 때 노안 탓 마세요 1분 해결법,핸드폰 화면이 갑자기 어두워져서 답답하셨던 적 있으신가요 많은 세대분들이 이를 ...,2026. 2. 10. 10:11,"['핸드폰', '화면', '갑자기', '어두워지다', '답답하다', '가요', '많..."
2,일월도서관 5060세대를 위한 인문학과 독서,프로그램명 세대를 위한 인문학과 독서 교육기간 매주 화 ...,2026. 2. 6. 15:16,"['프로그램', '세대', '위', '인문학', '독서', '교육', '기간', '..."
3,5060 핸드폰 화면 어두울 때 노안 탓 마세요 1분 해결법,핸드폰 화면이 갑자기 어두워져서 답답하셨던 적 있으신가요 많은 세대분들이 이를 ...,2026. 2. 10. 10:11,"['핸드폰', '화면', '갑자기', '어두워지다', '답답하다', '가요', '많..."
4,"5060 태블릿 PC 스마트폰 차이, 눈 침침한데 아직도 폰만 보세요?",태블릿 스마트폰 차이 눈 침침한데 아직도 폰만 보세요 목차 시력 보호와 화면 크기...,2026. 2. 13. 16:20,"['태블릿', '스마트폰', '차이', '눈', '침침하다', '폰', '보다', ..."
...,...,...,...,...
67769,시도해 보세요! 가을엔 이 코트 하나면 시니어패션 완성! #시니어패션 #가을코디 #...,"가을마다 옷 고민되신다면, 트렌치코트 하나로 해결!\n품격 있고 세련된 시니어룩을 ...",5개월 전,"['가을', '옷', '고민', '신다', '트렌치코트', '하나로', '해결', ..."
67770,와인 입문자는 필.수.시.청! 국가대표 소믈리에 송기범의 [와인 기초 이론]ㅣ콜키지...,뭔가 소믈리에님은 로제와인을 되게 좋아하시는듯 하네요 ㅋㅋㅋ,1년 전,"['뭔가', '소믈리에', '로', '와인', '좋아하다']"
67771,와인 입문자는 필.수.시.청! 국가대표 소믈리에 송기범의 [와인 기초 이론]ㅣ콜키지...,국기대표 송소믈리에님👍👍👍\n쏙쐬강의👍👍👍\n역쉬 미남 현대백화점 소믈리에님\n멋지...,4년 전,"['국기', '대표', '송', '소믈리에', '쏙', '쐬다', '강의', '역쉬..."
67772,와인 입문자는 필.수.시.청! 국가대표 소믈리에 송기범의 [와인 기초 이론]ㅣ콜키지...,이해 잘되게 잘 설명해주셔서 좋아용 >_< 송쏘믈리에님,4년 전,"['이해', '자다', '설명', '좋다', '송쏘믈리에님']"


In [16]:
from ast import literal_eval
import numpy as np

# 1. 'token' 컬럼의 문자열을 리스트 객체로 변환
# Replace NaN values with empty lists before applying literal_eval
df['token'] = df['token'].replace({np.nan: '[]'}).apply(literal_eval)

# 확인: 첫 번째 행의 token 타입 확인
print(type(df.loc[0, 'token']))  # <class 'list'>
print(df.loc[0, 'token'][0])     # 리스트이므로 첫 번째 요소 접근 가능

<class 'list'>
프로그램


In [18]:
%pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 102.8 MB/s eta 0:00:00


In [19]:
import gensim
from gensim.models.doc2vec import TaggedDocument
from gensim.models import Doc2Vec

tagged_corpus_list = []
for index, row in df.iterrows():
  text = row['token']
  tag = f'document {index}'
  tagged_corpus_list.append(TaggedDocument(tags = [tag], words = text))

d2v_model = Doc2Vec(vector_size=100, window = 5, dm=1)
d2v_model.build_vocab(tagged_corpus_list)
d2v_model.train(tagged_corpus_list, total_examples=d2v_model.corpus_count, epochs = 30)

df['vector'] = [d2v_model.dv.get_vector(f'document {i}') for i in df.index]
df

,title,contents,date,token,vector
0,일월도서관 5060세대를 위한 인문학과 독서,프로그램명 세대를 위한 인문학과 독서 교육기간 매주 화 ...,2026. 2. 6. 15:16,"[프로그램, 세대, 위, 인문학, 독서, 교육, 기간, 총, 회, 장소, 일월, 도...","[0.2153398, -0.34769863, 0.12321576, 0.9746835..."
1,5060 핸드폰 화면 어두울 때 노안 탓 마세요 1분 해결법,핸드폰 화면이 갑자기 어두워져서 답답하셨던 적 있으신가요 많은 세대분들이 이를 ...,2026. 2. 10. 10:11,"[핸드폰, 화면, 갑자기, 어두워지다, 답답하다, 가요, 많다, 세대, 이르다, 시...","[1.269576, -0.58685917, 1.9391439, 1.87285, -3..."
2,일월도서관 5060세대를 위한 인문학과 독서,프로그램명 세대를 위한 인문학과 독서 교육기간 매주 화 ...,2026. 2. 6. 15:16,"[프로그램, 세대, 위, 인문학, 독서, 교육, 기간, 총, 회, 장소, 일월, 도...","[0.20646186, -0.2949324, 0.13378023, 1.1109685..."
3,5060 핸드폰 화면 어두울 때 노안 탓 마세요 1분 해결법,핸드폰 화면이 갑자기 어두워져서 답답하셨던 적 있으신가요 많은 세대분들이 이를 ...,2026. 2. 10. 10:11,"[핸드폰, 화면, 갑자기, 어두워지다, 답답하다, 가요, 많다, 세대, 이르다, 시...","[1.1184816, -0.62760496, 2.0428095, 2.0506613,..."
4,"5060 태블릿 PC 스마트폰 차이, 눈 침침한데 아직도 폰만 보세요?",태블릿 스마트폰 차이 눈 침침한데 아직도 폰만 보세요 목차 시력 보호와 화면 크기...,2026. 2. 13. 16:20,"[태블릿, 스마트폰, 차이, 눈, 침침하다, 폰, 보다, 목차, 시력, 보호, 화면...","[2.7985396, -2.098159, 0.060710907, 2.3481069,..."
...,...,...,...,...,...
67769,시도해 보세요! 가을엔 이 코트 하나면 시니어패션 완성! #시니어패션 #가을코디 #...,"가을마다 옷 고민되신다면, 트렌치코트 하나로 해결!\n품격 있고 세련된 시니어룩을 ...",5개월 전,"[가을, 옷, 고민, 신다, 트렌치코트, 하나로, 해결, 품격, 세련되다, 시니어,...","[0.49317262, -0.8486047, 2.8212464, 2.9295716,..."
67770,와인 입문자는 필.수.시.청! 국가대표 소믈리에 송기범의 [와인 기초 이론]ㅣ콜키지...,뭔가 소믈리에님은 로제와인을 되게 좋아하시는듯 하네요 ㅋㅋㅋ,1년 전,"[뭔가, 소믈리에, 로, 와인, 좋아하다]","[-0.031123253, 0.10347265, 0.027223196, 0.1494..."
67771,와인 입문자는 필.수.시.청! 국가대표 소믈리에 송기범의 [와인 기초 이론]ㅣ콜키지...,국기대표 송소믈리에님👍👍👍\n쏙쐬강의👍👍👍\n역쉬 미남 현대백화점 소믈리에님\n멋지...,4년 전,"[국기, 대표, 송, 소믈리에, 쏙, 쐬다, 강의, 역쉬, 미남, 현대, 백화점, ...","[-0.37236947, 0.078326195, -0.15096982, -0.201..."
67772,와인 입문자는 필.수.시.청! 국가대표 소믈리에 송기범의 [와인 기초 이론]ㅣ콜키지...,이해 잘되게 잘 설명해주셔서 좋아용 >_< 송쏘믈리에님,4년 전,"[이해, 자다, 설명, 좋다, 송쏘믈리에님]","[0.09742288, -0.032804206, -0.042143792, 0.119..."


In [20]:
import pickle
with open('merged_result_from_csv.pkl','wb') as file:
    pickle.dump(df,file)

In [21]:
from scipy.cluster.hierarchy import dendrogram, linkage
import matplotlib.pyplot as plt
import numpy as np
X = np.vstack(df['vector'].values)

#dendrogram 시각화로 적절한 클러스터 사이즈를 예측해본다
#method="ward" -> 거리가 아닌 밀도기반
linked = linkage(X, method="ward")
plt.figure(figsize=(15,9))
dendrogram(linked)
plt.show()

KeyboardInterrupt: 

In [ ]:
from sklearn.metrics import silhouette_score
from sklearn.cluster import AgglomerativeClustering
import matplotlib.pyplot as plt
from tqdm import tqdm

# try 2-10 clusters and visualize score
scores = []
for i in tqdm(range(2,11)):
  agg = AgglomerativeClustering(n_clusters = i, linkage = 'ward')
  prediction = agg.fit_predict(list(df['vector']))
  score = silhouette_score(list(df['vector']),prediction)
  scores.append(score)

plt.plot(range(2,11),scores)

  0%|          | 0/9 [00:00<?, ?it/s]

In [ ]:
#clustering
agg = AgglomerativeClustering(n_clusters = 5, linkage = 'ward')
df['cluster'] = agg.fit_predict(list(df['vector']))
df